# Weather Prediction for New York City (JFK)

## 1. Problem Statement

The goal of this project is to predict future weather conditions for New York City
using historical meteorological data collected at JFK Airport.

After exploratory analysis, the task was split into two complementary problems:

1. **Regression** — predicting the daily maximum temperature (`TMAX`)
2. **Classification** — predicting the overall weather type  
   (`Sunny`, `Rainy`, `Snowy`, `Foggy`, `Stormy`)

Two separate models were trained for these tasks and later combined into a unified
inference pipeline that produces both predictions simultaneously.

---

## 2. Dataset Overview

The dataset contains daily weather observations, including:
- temperature measurements (TMAX, TMIN, TAVG)
- precipitation and snow indicators
- wind speed
- categorical weather flags
- temporal information (date)

The data spans multiple years, exhibiting:
- strong seasonal patterns
- temporal autocorrelation
- class imbalance in weather events (Sunny dominates)

All data was sorted chronologically and treated as a time series.

---

## 3. Data Preparation and Validation

Before modeling, the following steps were performed:

- parsing and validating date fields
- handling missing values based on column semantics
- removing unused or redundant columns
- verifying physical plausibility of measurements
- enforcing strict time ordering to prevent information leakage

---

## 4. Feature Engineering Strategy

### Time-Series Features
To capture temporal dependencies without leakage:
- lagged values (e.g. `TMAX_lag_1`)
- rolling statistics computed **only from past values**
- seasonal encodings (`month`, `dayofyear`)

All rolling features were shifted by one day to prevent using current-day information.

---

## 5. Regression Model: Maximum Temperature (TMAX)

### Feature Selection (Regression)

The following features were used:

- `TMIN` — strong physical correlation with TMAX
- `PRCP`, `SNOW` — weather conditions influencing temperature
- `AWND` — wind-driven cooling effects
- `TMAX_lag_1` — short-term autocorrelation
- `TMAX_roll_7` — local temperature trend
- `month`, `dayofyear` — seasonal effects

The feature `TAVG` was intentionally excluded to avoid information leakage,
as it is derived from `TMAX`.

### Model Choice

An **XGBoost Regressor** was selected due to its:
- ability to model nonlinear relationships
- robustness to mixed feature scales
- strong performance on tabular time-series data

A linear regression model was used as a baseline for comparison.

---

## 6. Classification Model: Weather Type

### Target Construction

The `weather_type` label was derived from meteorological signals
(precipitation, snow, wind, visibility conditions).
Explicit weather flag columns (WT-codes) were excluded from training
to avoid trivial rule-based prediction.

### Feature Selection (Classification)

Features used for classification:

- `PRCP`, `PRCP_roll_3`, `PRCP_roll_7` — short and medium-term precipitation
- `SNOW` — snowfall indicator
- `AWND` — wind intensity
- `TMAX_lag_1`, `TMAX_roll_7` — recent temperature context
- `month`, `dayofyear` — seasonal patterns

This feature set forces the model to learn meaningful physical relationships
instead of relying on explicit condition flags.

### Model Choice

An **XGBoost Classifier** was used with macro-averaged metrics to handle
strong class imbalance. Target labels were encoded using `LabelEncoder`,
which is saved alongside the model for inference.

---

Each file has a single responsibility, separating experimentation,
model logic, evaluation, and inference.

---

## 7. Model Evaluation and Results

### Regression (TMAX)
- MAE ≈ **3.4°F**
- RMSE ≈ **4.4°F**
- R² ≈ **0.94**

XGBoost improved upon the linear baseline by capturing nonlinear temperature dynamics
and reducing error on extreme days.

### Classification (Weather Type)
- Accuracy ≈ **0.82**
- Macro F1-score ≈ **0.65**

Most misclassifications occurred because of class disbalance

Attempts to rebalance classes via sample weighting were evaluated but ultimately
reverted, as they reduced overall model stability.

---

## 8. Unified Inference

A single prediction interface combines both models and returns:

- predicted maximum temperature
- predicted weather type
- optional class probabilities

---

## 9. Prediction example



In [ ]:
import pandas as pd
from preprocess.data_prep import clean_weather_data
from preprocess.features import build_features
from models.inference.predict import predict_weather_and_temperature

In [ ]:
df = pd.read_csv("../data/raw/JFK Airport Weather Data.csv")
df["DATE"] = pd.to_datetime(df["DATE"])

df = clean_weather_data(df)
df = build_features(df)

df = df.sort_values("DATE").reset_index(drop=True)

In [ ]:
df_infer = df.tail(500)


predictions = predict_weather_and_temperature(
    df_infer,
    return_weather_proba=True
)


In [ ]:
predictions[
    ["predicted_TMAX", "weather_type", "Sunny", "Rainy", "Snowy"]
].head()



,predicted_TMAX,weather_type,Sunny,Rainy,Snowy
19589,79.707367,Sunny,0.862021,0.000291,0.000018
19590,81.012215,Sunny,0.715697,0.000313,0.000026
19591,84.101089,Sunny,0.701651,0.000475,0.000033
19592,83.989151,Rainy,0.000335,0.588398,0.000028
19593,83.433784,Sunny,0.575503,0.000529,0.000036
